In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/Othercomputers/My Laptop/Google Drive/HUST/Lesson/Retriever

/content/drive/Othercomputers/My Laptop/Google Drive/HUST/Lesson/Retriever


## 🐍 Phần 2: Chuẩn Bị Dữ Liệu (20 phút)

### 2.1 Crawl Dữ Liệu Tiếng Việt

Sử dụng dataset Wikipedia tiếng Việt có sẵn trên huggingface

In [3]:
import json
from datasets import load_dataset

# Load Wikipedia tiếng Việt
dataset = load_dataset("vietgpt/wikipedia_vi")

# Chọn split
split = "train" if "train" in dataset else list(dataset.keys())[0]
print("Using split:", split)
print("Columns:", dataset[split].column_names)

# Lấy 100 bài đầu tiên
documents = []
for idx, doc in enumerate(dataset[split]):
    if idx >= 1000:
        break

    documents.append({
        "id": doc.get("id", f"doc_{idx}"),
        "title": doc.get("title"),
        "text": doc.get("text"),
        "url": doc.get("url"),
    })

# Lưu ra file JSON
with open("vietnamese_docs.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {len(documents)} documents")
print("Sample title:", documents[0]["title"])

README.md:   0%|          | 0.00/632 [00:00<?, ?B/s]

data/train-00000-of-00003-6218d2963e3020(…):   0%|          | 0.00/245M [00:00<?, ?B/s]

data/train-00001-of-00003-12e6c4fadbec91(…):   0%|          | 0.00/55.2M [00:00<?, ?B/s]

data/train-00002-of-00003-175fcfe1c45b0b(…):   0%|          | 0.00/270M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1284930 [00:00<?, ? examples/s]

Using split: train
Columns: ['id', 'revid', 'url', 'title', 'text']
✅ Saved 1000 documents
Sample title: Trang Chính


### 2.2 Xử Lý Dữ Liệu (Data Processing)

In [ ]:
!pip install langchain-text-splitters

In [5]:
import json
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

def clean_text(text: str) -> str:
    """Làm sạch text tiếng Việt"""
    text = re.sub(r"http[s]?://\S+", "", text)  # remove URLs
    text = re.sub(r"[^\w\s\u0100-\u01B0\u1EA0-\u1EFF]", " ", text)  # keep vi unicode ranges
    text = re.sub(r"\s+", " ", text).strip()
    return text

def make_splitter(chunk_size_chars=1500, chunk_overlap_chars=200):
    """
    RecursiveCharacterTextSplitter chunk theo ký tự.
    chunk_size_chars ~ 1200-2000 thường tương đương 200-350 từ VN (tuỳ văn bản).
    """
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size_chars,
        chunk_overlap=chunk_overlap_chars,
        separators=[
            "\n\n", "\n",
            ". ", "? ", "! ",  # sentence-ish
            "; ", ": ", ", ",
            " ", ""
        ],
        length_function=len,
        is_separator_regex=False,
    )

# ===== Load documents (input giữ nguyên) =====
with open("vietnamese_docs.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

splitter = make_splitter(chunk_size_chars=1500, chunk_overlap_chars=200)

processed_docs = []
doc_id = 0

for doc in documents:
    title = doc.get("title")
    url = doc.get("url")
    text = clean_text(doc.get("text", ""))

    if not text:
        continue

    chunks = splitter.split_text(text)

    for chunk in chunks:
        # Filter chunks quá ngắn (giữ logic tương tự bản cũ)
        if len(chunk.split()) > 10:
            processed_docs.append({
                "id": f"chunk_{doc_id}",
                "title": title,
                "content": chunk,
                "source_url": url
            })
            doc_id += 1

# ===== Save output (output giữ nguyên) =====
with open("processed_chunks.json", "w", encoding="utf-8") as f:
    json.dump(processed_docs, f, ensure_ascii=False, indent=2)

print(f"✅ Processed into {len(processed_docs)} chunks")
print("Sample chunk:", processed_docs[0]["content"][:200], "...")

✅ Processed into 9656 chunks
Sample chunk: Internet Society hay ISOC là một tổ chức quốc tế hoạt động phi lợi nhuận phi chính phủ và bao gồm các thành viên có trình độ chuyên ngành Tổ chức này chú trọng đến tiêu chuẩn giáo dục và các vấn đề về ...


### 2.3 Bài Tập 1: Kiểm Tra Dữ Liệu

**Task:** Viết code để:
1. Load file `processed_chunks.json`
2. In ra 3 chunks đầu tiên
3. Tính số lượng từ trung bình của mỗi chunk

**Gợi ý:** Dùng `json.load()` và string split

In [ ]:
# YOUR CODE HERE

Total chunks: 9656

--- Chunk 0 ---
Title: Internet Society
Content: Internet Society hay ISOC là một tổ chức quốc tế hoạt động phi lợi nhuận phi chính phủ và bao gồm các thành viên có trình độ chuyên ngành Tổ chức này chú trọng đến tiêu chuẩn giáo dục và các vấn đề về...

--- Chunk 1 ---
Title: Tiếng Việt
Content: Tiếng Việt cũng gọi là tiếng Việt Nam hay Việt ngữ là ngôn ngữ của người Việt và là ngôn ngữ chính thức tại Việt Nam Đây là tiếng mẹ đẻ của khoảng 85 dân cư Việt Nam cùng với hơn 4 triệu người Việt ki...

--- Chunk 2 ---
Title: Tiếng Việt
Content: nhà Lý với 6 thanh điệu Sau đó một số phụ âm đầu biến đổi cho tới ngày nay Trong quá trình biến đổi các phụ âm cuối rụng đi làm thay đổi các kết thúc âm tiết và phụ âm đầu chuyển từ lẫn lộn vô thanh v...

✅ Average words per chunk: 318.2


## 🧠 Phần 3: Embedding & Vector Generation (30 phút)

### 3.1 Chọn Embedding Model

**Multilingual-E5-Large** (khuyến nghị cho tiếng Việt):
- Hỗ trợ 100+ ngôn ngữ
- Chiều vector: 1024
- Chất lượng cao, nhưng cần GPU

**Cài đặt:**
```bash
pip install sentence-transformers
```

### 3.2 Generate Embeddings

In [7]:
from sentence_transformers import SentenceTransformer
import torch
import json

# Load model (nhẹ, nhanh cho T4)
model = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    device="cuda"
)

# FP16 để tăng tốc
model = model.half()
torch.backends.cuda.matmul.allow_tf32 = True

# Load chunks
with open("processed_chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# E5 prefix
texts = ["passage: " + chunk["content"] for chunk in chunks]

# Generate embeddings
print("Generating embeddings...")
embeddings = model.encode(
    texts,
    batch_size=128,          # nếu OOM → 64
    show_progress_bar=True,
    normalize_embeddings=True
)

# Gắn embeddings
for i, chunk in enumerate(chunks):
    chunk["embedding"] = embeddings[i].tolist()

# Lưu
with open("chunks_with_embeddings.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False)

print(f"✅ Generated embeddings for {len(chunks)} chunks")
print(f"Embedding dimension: {len(embeddings[0])}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Generating embeddings...


Batches:   0%|          | 0/76 [00:00<?, ?it/s]

✅ Generated embeddings for 9656 chunks
Embedding dimension: 384


### 3.3 Tính BM25 Scores

BM25 là thuật toán **full-text search** cổ điển, kết hợp tốt với vector search.

In [8]:
!pip install rank-bm25 underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.7 MB/s eta 0:00:00


In [9]:
from rank_bm25 import BM25Okapi
import json
import re
import pickle
from tqdm import tqdm

# =========================
# 1. Load chunks
# =========================
with open('chunks_with_embeddings.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

print(f"Chuẩn bị corpus từ {len(chunks)} chunks...")

# =========================
# 2. Tokenizer đơn giản
# =========================
def simple_tokenizer(text):
    # Xóa dấu câu, chuyển về chữ thường
    text = re.sub(r'[^\w\s]', '', text.lower())
    # Tách token bằng khoảng trắng
    return text.split()

print("Đang token hóa corpus (có thể mất vài giây)...")
corpus_tokens = [
    simple_tokenizer(chunk['content'])
    for chunk in tqdm(chunks)
]

# =========================
# 3. Train BM25
# =========================
print("\nĐang fit model BM25...")
bm25 = BM25Okapi(corpus_tokens)
print("Đã train xong!")

# =========================
# 4. Lưu BM25 index
# =========================
with open('bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25, f)

print("✅ BM25 index created")

Chuẩn bị corpus từ 9656 chunks...
Đang token hóa corpus (có thể mất vài giây)...


100%|██████████| 9656/9656 [00:01<00:00, 4867.28it/s]



Đang fit model BM25...
Đã train xong!
✅ BM25 index created


### 3.4 Bài Tập 2: Test Embedding Quality

**Task:**
1. Load embedding model
2. Encode 2 câu query tiếng Việt:
   - Query 1: "Trí tuệ nhân tạo là gì?"
   - Query 2: "AI là gì?"
3. Tính cosine similarity giữa 2 queries
4. Comment: 2 câu này tương đồng không?

**Gợi ý:** Dùng `cosine_similarity` từ sklearn

In [10]:
# YOUR CODE HERE
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('intfloat/multilingual-e5-small')

query1 = "Trí tuệ nhân tạo là gì?"
query2 = "AI là gì?"

# Encode queries
embedding1 = model.encode(query1, normalize_embeddings=True)
embedding2 = model.encode(query2, normalize_embeddings=True)

# Tính cosine similarity
similarity = util.pytorch_cos_sim(embedding1, embedding2)

print(f"Query 1: {query1}")
print(f"Query 2: {query2}")
print(f"✅ Cosine Similarity: {similarity[0][0]:.4f}")
print(f"Comment: {'Very similar' if similarity[0][0] > 0.8 else 'Similar' if similarity[0][0] > 0.5 else 'Different'}")

Query 1: Trí tuệ nhân tạo là gì?
Query 2: AI là gì?
✅ Cosine Similarity: 0.8909
Comment: Very similar


## 🗄️ Phần 4: Ingest Data vào Weaviate (25 phút)

### 4.1 Kết Nối Weaviate

In [11]:
!pip install weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 599.9/599.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 4.4 MB/s eta 0:00:00


In [13]:
import weaviate
import json
from weaviate.util import generate_uuid5

# URL từ Cloudflare của bạn (không bao gồm https:// ở phần host)
# URL Gốc: https://clearance-disabled-noted-objective.trycloudflare.com
HTTP_HOST = "c8538be6ece5.ngrok-free.app"
GRPC_HOST = "8.tcp.ngrok.io"

client = weaviate.connect_to_custom(
    http_host=HTTP_HOST,
    http_port=443,
    http_secure=True,

    grpc_host=GRPC_HOST,
    grpc_port=16999,
    grpc_secure=False,   # 🚨 QUAN TRỌNG: TCP tunnel → False

    # skip_init_checks=True  # bật nếu init bị timeout
)

# Kiểm tra connection
try:
    # Trong v4, chúng ta cần kết nối trước khi check
    client.connect()

    if client.is_live():
        print(f"✅ Weaviate is live!")
        # Lấy thêm thông tin meta (tuỳ chọn)
        meta = client.get_meta()
        print(f"ℹ️  Version: {meta.get('version')}")
    else:
        print("❌ Weaviate is NOT live")

except Exception as e:
    print(f"❌ Cannot connect to Weaviate: {e}")
    print("Kiểm tra:")
    print("1. Docker container có chạy không?")
    print("2. Cloudflare tunnel có đang hoạt động không?")
    print("3. Nếu lỗi gRPC, hãy thử thêm tham số 'skip_init_checks=True' trong config.")

finally:
    # Rất quan trọng ở v4: Phải đóng kết nối để giải phóng gRPC
    client.close()

✅ Weaviate is live!
ℹ️  Version: 1.32.22


### 4.2 Tạo Schema (Cấu Trúc Dữ Liệu)

In [16]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
# from weaviate.util import generate_uuid5  # vẫn dùng được nếu bạn cần UUID v5

client = weaviate.connect_to_custom(
    http_host=HTTP_HOST,
    http_port=443,
    http_secure=True,

    grpc_host=GRPC_HOST,
    grpc_port=16999,
    grpc_secure=False,  # 🚨 TCP tunnel → False (giống code cũ)
)

COLLECTION = "VietnameseDocument"

# Xóa collection cũ nếu có
try:
    client.collections.delete(COLLECTION)
    print("đã xóa")
except Exception:
    pass

# Tạo collection mới
# "vectorizer": "none" (v3)  ⇢  v4 dùng self_provided (bring your own vectors)
client.collections.create(
    name=COLLECTION,
    vector_config=Configure.Vectors.self_provided(),
    properties=[
        Property(
            name="title",
            data_type=DataType.TEXT,
            index_filterable=True,
            index_searchable=True,
        ),
        Property(
            name="content",
            data_type=DataType.TEXT,
            index_filterable=True,
            index_searchable=True,
        ),
        Property(
            name="source_url",
            data_type=DataType.TEXT,
            index_filterable=True,
            index_searchable=False,
        ),
        Property(
            name="chunk_id",
            data_type=DataType.TEXT,
            index_filterable=True,
            index_searchable=False,
        ),
    ],
)

print("✅ Collection created")

# (khuyến nghị) đóng client khi xong việc
# client.close()

đã xóa
✅ Collection created


### 4.3 Ingest Dữ Liệu

In [17]:
import json
import weaviate
from weaviate.util import generate_uuid5
from tqdm import tqdm

# Load chunks with embeddings
with open("chunks_with_embeddings.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

collection = client.collections.use("VietnameseDocument")

# Batch ingest + progress bar
with collection.batch.fixed_size(batch_size=10) as batch:
    for chunk in tqdm(
        chunks,
        desc="📥 Ingesting chunks",
        unit="chunk",
        ncols=100,
    ):
        properties = {
            "title": chunk["title"],
            "content": chunk["content"],
            "source_url": chunk["source_url"],
            "chunk_id": chunk["id"],
        }

        uuid = generate_uuid5(chunk["id"])

        batch.add_object(
            properties=properties,
            uuid=uuid,
            vector=chunk["embedding"],
        )

print(f"✅ Ingested {len(chunks)} documents into Weaviate")

# Kiểm tra total count (v4)
agg = collection.aggregate.over_all(total_count=True)
print(f"Total documents in DB: {agg.total_count}")

# Kiểm tra lỗi batch
failed = collection.batch.failed_objects
if failed:
    print(f"⚠️ Failed objects: {len(failed)}")
    print("First failed object:", failed[0])

client.close()

📥 Ingesting chunks: 100%|███████████████████████████████████| 9656/9656 [09:50<00:00, 16.36chunk/s]


✅ Ingested 9656 documents into Weaviate
Total documents in DB: 9656


### 4.4 Bài Tập 3: Query Kiểm Tra Dữ Liệu

**Task:**
1. Query Weaviate để lấy 5 documents (bất kỳ)
2. In ra title và content (truncate tới 100 ký tự)

**Gợi ý:** Dùng `.limit(5)`

In [18]:
import weaviate

client = weaviate.connect_to_custom(
    http_host=HTTP_HOST,
    http_port=443,
    http_secure=True,

    grpc_host=GRPC_HOST,
    grpc_port=16999,
    grpc_secure=False,  # 🚨 TCP tunnel → False
)

collection = client.collections.use("VietnameseDocument")

# Query get documents (v4)
response = collection.query.fetch_objects(
    limit=5
)

documents = response.objects

for i, obj in enumerate(documents):
    props = obj.properties

    print(f"\n--- Document {i+1} ---")
    print(f"Title: {props.get('title')}")
    print(f"Content: {props.get('content', '')[:100]}...")
    print(f"URL: {props.get('source_url')}")


--- Document 1 ---
Title: Việt Nam Cộng hòa
Content: Loan binh lính Mỹ Hàn Quốc Philippines Ngay cả Mặt trận Dân tộc Giải phóng cũng tận dụng thị trường ...
URL: https://vi.wikipedia.org/wiki?curid=98

--- Document 2 ---
Title: Chiến tranh Đông Dương
Content: khăn là kiểm soát được các bè phái chính trị khác nhau vì không phải tất cả thân Việt Minh mà tất cả...
URL: https://vi.wikipedia.org/wiki?curid=1287

--- Document 3 ---
Title: Thành phố Hồ Chí Minh
Content: trạng ngập lụt trong trung tâm thành phố đang ở mức báo động cao xảy ra cả trong mùa khô Diện tích k...
URL: https://vi.wikipedia.org/wiki?curid=39

--- Document 4 ---
Title: Nguyễn Trãi
Content: giáo sau Nguyễn Trãi là một văn hóa Đại Việt được cấu trúc theo mô hình Nho giáo từ Trung Quốc Nguyễ...
URL: https://vi.wikipedia.org/wiki?curid=1555

--- Document 5 ---
Title: Lạm phát
Content: bởi các thao túng dữ liệu kinh tế chẳng hạn như số liệu lạm phát và GDP cho lợi ích chính trị và giả...
URL: https://vi.wikipedia.org/wiki?c

## 🔍 Phần 5: Semantic Search (20 phút)

### 5.1 Cơ Bản Semantic Search

Tìm kiếm **dựa trên ý nghĩa** (không phải từ khóa chính xác).

In [19]:
import weaviate
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")

def semantic_search(query: str, top_k: int = 5):
    """Semantic search dùng embeddings (Weaviate v4)"""

    # Encode query
    query_embedding = model.encode(query, normalize_embeddings=True)  # numpy array

    collection = client.collections.use("VietnameseDocument")

    # Vector search
    response = collection.query.near_vector(
        near_vector=query_embedding.tolist(),
        limit=top_k,
        return_metadata=["distance"],  # tương đương with_additional(["distance"])
    )

    # Trả về list object (giống list docs)
    return response.objects

# Test
query = "Học máy và trí tuệ nhân tạo"
results = semantic_search(query, top_k=5)

print(f"Query: {query}\n")
for i, obj in enumerate(results):
    props = obj.properties
    dist = obj.metadata.distance  # distance nằm trong metadata

    print(f"{i+1}. [{dist:.4f}] {props.get('title')}")
    print(f"   {props.get('content', '')[:150]}...\n")


Query: Học máy và trí tuệ nhân tạo

1. [0.1364] Trung Quốc
   lái tàu khu trục tàu đổ bộ đệm không khí tàu ngầm tên lửa vệ tinh hệ thống vũ khí robotics trí tuệ nhân tạo bán dẫn ổ đĩa trạng thái rắn thông tin di ...

2. [0.1416] Máy tính
   thành hiện thực Máy tính không thể giải quyết tất cả mọi vấn đề của toán học Alan Turing đã sáng tạo ra khoa học lý thuyết máy tính trong đó đề cập tớ...

3. [0.1421] Khoa học máy tính
   là một ngành học riêng biệt với sự ra đời của các khoa Khoa học máy tính đầu tiên và các chương trình đào tạo đại học chuyên ngành Khoa học máy tính T...

4. [0.1467] Công nghệ thông tin
   kế cơ sở dữ liệu cũng như quản lý quản trị toàn bộ hệ thống Công nghệ thông tin bắt đầu lan rộng hơn nữa so với máy tính cá nhân và công nghệ mạng thô...

5. [0.1471] Trung Quốc
   Quốc hiện nay tính theo sức mua tương đương các công ty công nghệ của Nhật như Nikon Canon Sony và Panasonic đã có vị trí quan trọng trên thị trường q...



### 5.3 Bài Tập 4: Semantic Search với Queries Khác Nhau

**Task:**
1. Thực hiện semantic search với 3 queries:
   - "Python lập trình"
   - "Lịch sử Việt Nam"
   - "Công nghệ blockchain"
2. Với mỗi query, in ra top 3 results (title + distance)

**Gợi ý:** Dùng function từ 5.1, tạo loop qua 3 queries

In [ ]:
# Weaviate v4 version
# assumes you already have: client (v4), model (SentenceTransformer)

from tqdm import tqdm  # optional

queries = [
    "Python lập trình",
    "Lịch sử Việt Nam",
    "Công nghệ blockchain"
]

collection = client.collections.use("VietnameseDocument")

# TO DO
# Code here

## 🎯 Phần 6: Hybrid Search (BM25 + Vector) (25 phút)

### 6.1 BM25 Search


In [21]:
def bm25_search(query: str, top_k: int = 5):
    """Full-text search dùng BM25 (Weaviate v4)"""

    collection = client.collections.use("VietnameseDocument")

    resp = collection.query.bm25(
        query=query,
        limit=top_k,
        return_metadata=["score"],              # = with_additional(["score"])
        return_properties=["title", "content"], # lấy field cần dùng
    )

    return resp.objects

# Test
query = "Trí tuệ nhân tạo"
results = bm25_search(query, top_k=5)

print(f"BM25 Search: {query}\n")
for i, obj in enumerate(results):
    props = obj.properties
    score = obj.metadata.score

    print(f"{i+1}. [score: {score:.4f}] {props.get('title')}")
    print(f"   {props.get('content', '')[:100]}...\n")


BM25 Search: Trí tuệ nhân tạo

1. [score: 8.4547] Tổ chức sở hữu trí tuệ
   Tổ chức sở hữu trí tuệ là một tổ chức quốc tế liên chính phủ liên quan đến việc hợp tác trong các lĩ...

2. [score: 8.1107] Google
   Google đã mua lại Waze một hợp đồng trị giá 966 triệu đô la Mặc dù Waze sẽ vẫn là một thực thể độc l...

3. [score: 7.8929] Trung Quốc
   lái tàu khu trục tàu đổ bộ đệm không khí tàu ngầm tên lửa vệ tinh hệ thống vũ khí robotics trí tuệ n...

4. [score: 7.2978] Tiến quân ca
   bỏ tiền ra sản xuất nên họ là chủ sở hữu hợp pháp của bản ghi ai muốn dùng đều phải xin phép Tức là ...

5. [score: 7.0229] VietNamNet
   bài về xây khách sạn trong công viên Thống Nhất Chính phủ ra quyết định huỷ bỏ dự án Loạt bài phản b...



### 6.2 Hybrid Search: Kết Hợp BM25 + Vector

Kết hợp cả **keyword matching** (BM25) và **semantic meaning** (vector):

In [22]:
def hybrid_search(query: str, top_k: int = 5, alpha: float = 0.5):
    """
    Hybrid search = BM25 + Vector (Weaviate v4)
    alpha: 0.0 (pure BM25) ~ 1.0 (pure vector)
    """

    # ✅ encode + normalize mặc định
    query_embedding = model.encode(query, normalize_embeddings=True)

    collection = client.collections.use("VietnameseDocument")

    resp = collection.query.hybrid(
        query=query,
        vector=query_embedding.tolist(),
        alpha=alpha,
        limit=top_k,
        return_metadata=["score"],          # = with_additional(["score"])
        return_properties=["title"],        # lấy field cần in
    )

    return resp.objects


# Test với alpha=0.5 (balanced)
query = "Machine learning ứng dụng"
results = hybrid_search(query, top_k=5, alpha=0.5)

print(f"Hybrid Search (alpha=0.5): {query}\n")
for i, obj in enumerate(results):
    score = obj.metadata.score
    title = obj.properties.get("title")
    print(f"{i+1}. [score: {score:.4f}] {title}")

Hybrid Search (alpha=0.5): Machine learning ứng dụng

1. [score: 0.5433] Khoa học ứng dụng
2. [score: 0.5209] Toán học ứng dụng
3. [score: 0.5000] Java Platform, Standard Edition
4. [score: 0.5000] Tin sinh học
5. [score: 0.4983] Java (công nghệ)


### 6.3 Bài Tập 5: Tuning Alpha Parameter

**Task:**
1. Test hybrid search với 3 giá trị alpha: 0.3, 0.5, 0.7
2. Query: "Khoa học dữ liệu"
3. So sánh top 1 result của mỗi alpha
4. Nhận xét: alpha nào cho kết quả tốt nhất?

**Gợi ý:** Tạo loop qua alpha values


In [23]:
# Weaviate v4 (giả sử `client` và `model` đã có sẵn)

query = "Khoa học dữ liệu"
alphas = [0.3, 0.5, 0.7]

# ✅ encode + normalize mặc định (theo yêu cầu của bạn)
# TO DO
# Code here

print("\n✅ Comment: Chọn alpha nào cho kết quả tốt nhất?")

Query: Khoa học dữ liệu

Alpha=0.3: Khoa học Trái Đất (score: 0.9790)
Alpha=0.5: Khoa học Trái Đất (score: 0.9649)
Alpha=0.7: Khoa học Trái Đất (score: 0.9509)

✅ Comment: Chọn alpha nào cho kết quả tốt nhất?


## 🎖️ Phần 7: Rerank Search (20 phút)

### 7.1 Tại Sao Cần Reranking?

- **Hybrid search** lấy top K kết quả
- **Reranker** sắp xếp lại những results này dựa trên **semantic relevance** sâu hơn
- Cross-encoder (reranker) thường cho kết quả chính xác hơn sentence-transformer

### 7.2 Cài Đặt Cross-Encoder Reranker

```python
pip install sentence-transformers
```

### 7.3 Implement Reranker

In [25]:
from sentence_transformers import CrossEncoder

# Load reranker (hỗ trợ tiếng Việt)
reranker = CrossEncoder(
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    max_length=512
)

def hybrid_search_with_rerank(query, top_k=5, num_candidates=20):
    """
    Hybrid search + Reranking (Weaviate v4)
    1. Lấy num_candidates từ hybrid search
    2. Rerank và trả về top_k
    """

    # ✅ encode + normalize mặc định
    query_embedding = model.encode(query, normalize_embeddings=True)

    collection = client.collections.use("VietnameseDocument")

    # Lấy candidates (nhiều hơn top_k)
    resp = collection.query.hybrid(
        query=query,
        vector=query_embedding.tolist(),
        alpha=0.5,
        limit=num_candidates,
        return_properties=["title", "content"],  # cần để rerank + in output
    )

    candidates = resp.objects

    if not candidates:
        return []

    # Rerank: tính score cho mỗi (query, document) pair
    pairs = [(query, obj.properties.get("content", "")) for obj in candidates]
    rerank_scores = reranker.predict(pairs)

    # Sắp xếp theo rerank score
    scored = list(zip(candidates, rerank_scores))
    scored.sort(key=lambda x: x[1], reverse=True)

    # Trả về top_k: giữ format gần giống code cũ (dict properties)
    return [({"title": obj.properties.get("title"),
              "content": obj.properties.get("content", "")}, float(score))
            for obj, score in scored[:top_k]]


# Test
query = "Ứng dụng AI trong y tế"
results = hybrid_search_with_rerank(query, top_k=5, num_candidates=20)

print(f"Query: {query}\n")
for i, (doc, score) in enumerate(results):
    print(f"{i+1}. [rerank score: {score:.4f}] {doc['title']}")
    print(f"   {doc['content'][:100]}...\n")

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Query: Ứng dụng AI trong y tế

1. [rerank score: -1.4612] Tia X
   gọi là Khoa chẩn đoán hình ảnh Việc sử dụng tia X đặc biệt hữu dụng trong việc xác định bệnh lý về x...

2. [rerank score: -1.8728] Công nghệ nano
   những hình ảnh của phân tử và nguyên tử của vật chất Những phương tiện dụng cụ khác bao gồm Điều chế...

3. [rerank score: -2.1289] Tin học
   tin thúc đẩy mối quan hệ giữa khoa học thư viện và phát triển khoa học thông tin để mang lại lợi ích...

4. [rerank score: -2.7382] Tin sinh học
   được dùng để tăng tốc độ hoặc giúp tự động hoàn toàn quá trình xử lý định lượng và phân tích một lượ...

5. [rerank score: -2.7846] Y học
   nói trên người thầy thuốc có thể quyết định điệu trị ngay hoặc đề nghị một số xét nghiệm cận lâm sàn...



In [26]:
client.close()